<a href="https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/05_transmil_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - TransMIL Training on Real PANDA Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/05_transmil_training.ipynb)

**Role:** Person 3 (Himesh)
**Model:** TransMIL (Transformer-based Correlated Multiple Instance Learning)
**Task:** Train TransMIL on real PANDA features across 3 seeds (0, 1, 2) and evaluate against the baseline.

All datasets and checkpoints are stored directly in **Google Drive in the cloud** — zero large files are stored on your local laptop.


## 1. Setup & Environment
Clone the repository and install lightweight dependencies quietly without conflicting with Colab's pre-installed packages.


In [ ]:
# Setup repository and dependencies quietly
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    if not os.path.exists("prostate-mil-project"):
        !git clone https://github.com/koushikkkkkkkkkk/prostate-mil-project.git
        %cd prostate-mil-project
    else:
        %cd prostate-mil-project
        !git pull origin main
    !pip install -q pyyaml
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")

import torch
print(f"Running in Colab: {IN_COLAB}")
print(f"Working directory: {ROOT}")
print(f"PyTorch: {torch.__version__} | GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


## 2. Mount Google Drive & Link Features
All 420 slide feature files (~600 MB) and training outputs will stay safely in your Google Drive cloud storage.


In [ ]:
# Mount Google Drive (keeps all 600 MB of features safely in the cloud across sessions)
import shutil

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        
        # Check candidate Google Drive folders where features might already be saved
        DRIVE_CANDIDATES = [
            Path("/content/drive/MyDrive/prostate_mil_data"),
            Path("/content/drive/MyDrive/panda_mil_data"),
            Path("/content/drive/MyDrive/panda_mil_project"),
            Path("/content/drive/MyDrive/prostate_mil"),
        ]
        
        GDRIVE_DIR = None
        for candidate in DRIVE_CANDIDATES:
            cand_feat = candidate / "features"
            if cand_feat.exists() and len(list(cand_feat.glob("*.pt"))) >= 420:
                GDRIVE_DIR = candidate
                break
        
        if GDRIVE_DIR is None:
            GDRIVE_DIR = Path("/content/drive/MyDrive/prostate_mil_data")
            
        FEATURES_DIR = GDRIVE_DIR / "features"
        BACKUP_DIR = GDRIVE_DIR / "results"
        FEATURES_DIR.mkdir(parents=True, exist_ok=True)
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        
        # Symlink data/features -> Google Drive so any command path works seamlessly
        local_feat = ROOT / "data/features"
        if local_feat.exists() and not local_feat.is_symlink():
            shutil.rmtree(local_feat)
        if not local_feat.exists():
            local_feat.symlink_to(FEATURES_DIR)
        print(f"Google Drive mounted and linked: {FEATURES_DIR}")
    except Exception as e:
        print(f"Google Drive mount notice ({e}). Using Colab local storage.")
        FEATURES_DIR = ROOT / "data/features"
        BACKUP_DIR = ROOT / "results"
else:
    FEATURES_DIR = ROOT / "data/features"
    BACKUP_DIR = ROOT / "results"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Features storage directory: {FEATURES_DIR}")


## 3. Verify & Download Real PANDA Features
We download pre-extracted 1024-D ResNet50 patch embeddings for our 420 slides (300 train, 60 val, 60 test) from the Hugging Face computational pathology repository.
* Total download: ~600 MB (takes ~1-2 min on Colab's high-speed cloud connection).
* Once downloaded to Google Drive, future runs find them cached instantly!


In [ ]:
# Verify and auto-download real ResNet50 features directly to Google Drive
MANIFEST = ROOT / "data/common_split.csv"
existing = list(FEATURES_DIR.glob("*.pt"))
print(f"Status: {len(existing)} / 420 feature files currently present.")

if len(existing) < 420:
    print("Downloading real ResNet50 features directly into Google Drive...")
    !python scripts/download_real_features.py --manifest {MANIFEST} --out-dir "{FEATURES_DIR}" --workers 16
    existing = list(FEATURES_DIR.glob("*.pt"))
    print(f"Download complete: {len(existing)} / 420 feature files ready.")
else:
    print("All 420 real slide features are ready in Google Drive!")


## 4. Model Architecture & Sanity Check
Verify TransMIL parameters and tensor shapes with a dummy bag before running training.


In [ ]:
import yaml
from src.models import build_model, count_parameters

CONFIG = ROOT / "configs/transmil.yaml"
OUT = ROOT / "results/metrics/transmil"
OUT.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(CONFIG.read_text())["model"]
cfg["in_dim"] = 1024
model = build_model("transmil", cfg)
model.eval()

dummy_bag = torch.randn(1, 64, cfg["in_dim"])
with torch.no_grad():
    out = model(dummy_bag)

print(f"Model Architecture: TransMIL (Simplified)")
print(f"Total Parameters:   {count_parameters(model):,}")
print(f"Logits shape:       {tuple(out['logits'].shape)} (batch_size, n_classes)")
print(f"Attention shape:    {tuple(out['attention'].shape)} (batch_size, n_patches)")
print(f"Attention sum:      {float(out['attention'].sum()):.4f}")


## 5. Train TransMIL (3 Seeds: 0, 1, 2)
We train across seeds 0, 1, 2 on the locked  using Colab's GPU.
Each epoch streams progress in real time below. On a T4 GPU, training all 3 seeds takes ~4-6 minutes.


In [ ]:
# Train TransMIL across all 3 seeds with real-time output streaming
!python -m src.train --model transmil --manifest {MANIFEST} --features-root "{FEATURES_DIR}" --config {CONFIG} --out-dir {OUT} --seeds 0 1 2


## 6. Official TransMIL Results & Performance
Display the official test metrics, markdown summary table, and test-set confusion matrix.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

table_file = OUT / "table_transmil.md"
if table_file.exists():
    print("### Official TransMIL Results across 3 Seeds:")
    print(table_file.read_text())

agg_file = OUT / "aggregate_transmil.json"
if agg_file.exists():
    agg = json.loads(agg_file.read_text())
    print("
Per-seed summary:")
    display(pd.DataFrame(agg.get("per_run", [])))
    print("====================================================")
    print(f"TransMIL Mean Test QWK: {agg['qwk']['mean']:.4f} ± {agg['qwk']['std']:.4f}")
    print(f"Accuracy:              {agg['accuracy']['mean']:.4f}")
    print(f"Balanced Accuracy:     {agg['balanced_accuracy']['mean']:.4f}")
    print(f"Macro F1:              {agg['macro_f1']['mean']:.4f}")
    print("====================================================")

conf_pngs = sorted(OUT.glob("confusion_*.png"))
if conf_pngs:
    print(f"
Test Confusion Matrix ({conf_pngs[0].name}):")
    display(Image(filename=str(conf_pngs[0]), width=450))


## 7. Attention Behaviour & Distribution Analysis
Inspect TransMIL self-attention distributions and entropy across test slides to verify whether the transformer focuses on specific diagnostic tokens.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import FeatureBagDataset, read_manifest, split_records
from src.evaluate_clam_sb import load_checkpoint
from src.heatmaps import attention_entropy, normalised_attention_entropy

ckpt_path = OUT / "checkpoints/transmil_seed0_best.pt"
if ckpt_path.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, _ = load_checkpoint(ckpt_path, device=device)
    model.eval()

    records = split_records(read_manifest(MANIFEST), "test")
    dataset = FeatureBagDataset(records[:5], FEATURES_DIR, load_coords=False, train=False)

    fig, axes = plt.subplots(1, min(5, len(dataset)), figsize=(18, 3.5))
    if len(dataset) == 1:
        axes = [axes]

    print("TransMIL Attention entropy on test slides (Seed 0):")
    for i in range(min(5, len(dataset))):
        item = dataset[i]
        rec = dataset.records[i]
        feats = item["features"].to(device).unsqueeze(0)
        with torch.no_grad():
            out = model(feats)
        attn = out["attention"].squeeze(0).cpu().numpy()
        pred = int(out["logits"].argmax(dim=-1).squeeze(0).cpu())

        ent = attention_entropy(attn)
        norm_ent = normalised_attention_entropy(attn)
        print(f"Slide {rec.image_id[:10]}... | True ISUP: {rec.label} | Pred ISUP: {pred} | Patches: {len(attn)} | Entropy: {ent:.3f}")

        ax = axes[i]
        ax.hist(attn, bins=30, color="#2ca02c", edgecolor="white")
        ax.set_title(f"True: {rec.label} | Pred: {pred}
{rec.image_id[:8]}...", fontsize=10)
        ax.set_xlabel("Attention")
        if i == 0:
            ax.set_ylabel("Patch count")

    plt.tight_layout()
    plt.show()
else:
    print(f"Checkpoint {ckpt_path} not found. Run training cell first.")


## 8. Cloud Backup to Google Drive
Permanently backs up all metrics, checkpoint weights, and figures to Google Drive.


In [ ]:
# Backup TransMIL results permanently to Google Drive
if IN_COLAB and GDRIVE_DIR.exists():
    target_backup = BACKUP_DIR / "transmil"
    if target_backup.exists():
        shutil.rmtree(target_backup)
    shutil.copytree(OUT, target_backup)
    print(f"TransMIL results permanently backed up to Google Drive at: {target_backup}")
else:
    print(f"Results stored locally at: {OUT}")
